# Algorithm Testing: UCS vs A*-Inspired Budgeted Crawler

This notebook tests the search modules on a small directed graph. It does not load or rebuild the SNAP dataset.

Run it from the project root (`Intelligent-Web-Crawler/`) so that `src` can be imported.


In [1]:
import sys
from pathlib import Path

# If this notebook is opened from notebooks/, add the project root.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.ucs import run_ucs
from src.astar import run_astar
from src.crawler import run_crawler


## 1. Create a small directed graph

The matrix uses `adjacency[u, v] = 1` to mean that webpage `u` links to webpage `v`. The example includes multiple paths, a cycle, different node degrees and a disconnected node.


In [2]:
import numpy as np

# Node indices: 0 through 7. Original SNAP-like IDs are kept separately.
node_ids = [10, 20, 30, 40, 50, 60, 70, 80]

edges = [
    (0, 1), (0, 2),
    (1, 3), (1, 4),
    (2, 4), (2, 5),
    (3, 6), (4, 1),  # cycle: 1 -> 4 -> 1
    (4, 6), (5, 6),
]

adjacency = np.zeros((len(node_ids), len(node_ids)), dtype=np.int8)
for source, destination in edges:
    adjacency[source, destination] = 1

in_degree = adjacency.sum(axis=0)
out_degree = adjacency.sum(axis=1)

# Dummy scores for independent algorithm testing only.
# In the actual project, use Person 1's precomputed PageRank values.
pagerank = np.array([0.05, 0.10, 0.08, 0.12, 0.30, 0.07, 0.25, 0.03])

graph_data = {
    "adjacency": adjacency,
    "node_ids": node_ids,
    "node_to_index": {node_id: i for i, node_id in enumerate(node_ids)},
    "in_degree": in_degree,
    "out_degree": out_degree,
    "pagerank": pagerank,
}

print("Nodes:", len(node_ids))
print("Directed edges:", len(edges))
print("Disconnected node index: 7")


Nodes: 8
Directed edges: 10
Disconnected node index: 7


## 2. Run both algorithms with the same start node and budget

In [11]:
start_node = 0
budget = 5

ucs_result = run_ucs(graph_data, start_node, budget)
astar_result = run_astar(graph_data, start_node, budget)

print("UCS")
for key, value in ucs_result.items():
    print(f"{key}: {value}")

print("\nA*-inspired")
for key, value in astar_result.items():
    print(f"{key}: {value}")


UCS
visited_node_indices: [0, 1, 2, 3, 4]
visited_node_ids: [10, 20, 30, 40, 50]
discovered_node_indices: [0, 1, 2, 3, 4, 5, 6]
total_cost: 6
execution_time: 0.00018790000012813834

A*-inspired
visited_node_indices: [0, 1, 2, 4, 3]
visited_node_ids: [10, 20, 30, 50, 40]
discovered_node_indices: [0, 1, 2, 3, 4, 5, 6]
total_cost: 6
execution_time: 0.00015630000052624382


## 3. Verify the shared requirements

These checks confirm the budget, uniqueness, start-node inclusion, original-ID mapping and directed-edge behavior.


In [4]:
def verify_result(result, graph_data, start_node, budget):
    visited = result["visited_node_indices"]
    discovered = result["discovered_node_indices"]

    assert len(visited) <= budget
    assert len(visited) == len(set(visited)), "A node was visited more than once."
    assert len(discovered) == len(set(discovered)), "A node was discovered more than once."
    assert visited[0] == start_node, "The start node must be visited first."
    assert result["visited_node_ids"] == [graph_data["node_ids"][i] for i in visited]
    assert result["total_cost"] >= 0
    assert result["execution_time"] >= 0

    # Every visited node except the start must be reachable from an earlier
    # visited node. This also checks that traversal follows outgoing edges.
    visited_so_far = {start_node}
    for node in visited[1:]:
        assert any(graph_data["adjacency"][parent, node] != 0 for parent in visited_so_far), (
            f"Node {node} was visited without an outgoing path from prior visits."
        )
        visited_so_far.add(node)

    return "All checks passed."


print("UCS:", verify_result(ucs_result, graph_data, start_node, budget))
print("A* :", verify_result(astar_result, graph_data, start_node, budget))


UCS: All checks passed.
A* : All checks passed.


## 4. Test different budgets

The start node counts as visit number one. A budget of one therefore visits only the start node.


In [5]:
for test_budget in [1, 2, 3, 5, 8, 20]:
    ucs = run_ucs(graph_data, start_node, test_budget)
    astar = run_astar(graph_data, start_node, test_budget)
    print(
        f"Budget={test_budget:2d} | "
        f"UCS visited={len(ucs['visited_node_indices'])}, "
        f"A* visited={len(astar['visited_node_indices'])}"
    )


Budget= 1 | UCS visited=1, A* visited=1
Budget= 2 | UCS visited=2, A* visited=2
Budget= 3 | UCS visited=3, A* visited=3
Budget= 5 | UCS visited=5, A* visited=5
Budget= 8 | UCS visited=7, A* visited=7
Budget=20 | UCS visited=7, A* visited=7


## 5. Test a simple chain

A chain should produce the same visit order for UCS and A*-inspired exploration when only one next node is available at each step.


In [6]:
chain_adjacency = np.zeros((4, 4), dtype=np.int8)
chain_adjacency[0, 1] = 1
chain_adjacency[1, 2] = 1
chain_adjacency[2, 3] = 1

chain_data = {
    "adjacency": chain_adjacency,
    "node_ids": [101, 102, 103, 104],
    "node_to_index": {101: 0, 102: 1, 103: 2, 104: 3},
    "in_degree": chain_adjacency.sum(axis=0),
    "out_degree": chain_adjacency.sum(axis=1),
    "pagerank": np.array([0.4, 0.3, 0.2, 0.1]),
}

print("UCS:", run_ucs(chain_data, 0, 4)["visited_node_indices"])
print("A* :", run_astar(chain_data, 0, 4)["visited_node_indices"])


UCS: [0, 1, 2, 3]
A* : [0, 1, 2, 3]


## 6. Check the high-PageRank, non-nearest webpage case

In this sample, node index 4 has the highest PageRank, but it is two hyperlinks away from the start. Nodes 1 and 2 are only one hyperlink away. The A*-inspired strategy should be able to prioritize node 4 once it is discovered, depending on its combined importance score.

In [7]:
astar_order = run_astar(graph_data, 0, 5)['visited_node_indices']
print('A*-inspired visit order:', astar_order)
assert 4 in astar_order and astar_order.index(4) < astar_order.index(3)
print('High-PageRank prioritization example passed.')

A*-inspired visit order: [0, 1, 2, 4, 3]
High-PageRank prioritization example passed.


## 7. Test a disconnected graph

The crawler must stop when its reachable frontier is empty, even if the requested budget is larger than the reachable component.


In [8]:
disconnected_result = run_ucs(graph_data, 7, 10)
print("Visited from disconnected node:", disconnected_result["visited_node_indices"])
assert disconnected_result["visited_node_indices"] == [7]
print("Disconnected-graph check passed.")


Visited from disconnected node: [7]
Disconnected-graph check passed.


## 8. Use the common crawler interface

In [9]:
print(run_crawler(graph_data, start_node, budget, algorithm="ucs"))
print(run_crawler(graph_data, start_node, budget, algorithm="astar"))


{'visited_node_indices': [0, 1, 2, 3, 4], 'visited_node_ids': [10, 20, 30, 40, 50], 'discovered_node_indices': [0, 1, 2, 3, 4, 5, 6], 'total_cost': 6, 'execution_time': 0.00015389999998660642}
{'visited_node_indices': [0, 1, 2, 4, 3], 'visited_node_ids': [10, 20, 30, 50, 40], 'discovered_node_indices': [0, 1, 2, 3, 4, 5, 6], 'total_cost': 6, 'execution_time': 0.0002214000001004024}


## 9. Test notes and interpretation

- UCS uses accumulated path cost. With every edge costing one, it explores by increasing hop distance.
- The A*-inspired version uses `f(n) = g(n) + h(n)`, with `h(n) = 2 * (1 - importance(n))`.
- Importance is a weighted combination of normalized PageRank, in-degree, out-degree and balanced connectivity.
- This heuristic is not an admissible estimate of distance to a goal. The crawler has no single goal webpage, so conventional A* optimality guarantees do not apply.
- A* may discover high-importance pages earlier, but it does not guarantee that the top-PageRank pages will all be visited within every budget.
- For the final SNAP experiment, replace only the sample `graph_data` with the dictionary supplied by Person 1. Do not calculate PageRank again here.


In [10]:

from scipy.sparse import csr_matrix

# Convert our small sample graph into CSR format
csr_graph_data = graph_data.copy()

csr_graph_data["adjacency"] = csr_matrix(adjacency)

# Run both algorithms using CSR adjacency
csr_ucs = run_ucs(csr_graph_data, 0, 5)
csr_astar = run_astar(csr_graph_data, 0, 5)

print("CSR UCS:", csr_ucs)
print("CSR A*:", csr_astar)

# Verify the results
assert len(csr_ucs["visited_node_indices"]) <= 5
assert len(csr_astar["visited_node_indices"]) <= 5

assert len(set(csr_ucs["visited_node_indices"])) == len(
    csr_ucs["visited_node_indices"]
)

assert len(set(csr_astar["visited_node_indices"])) == len(
    csr_astar["visited_node_indices"]
)

print("CSR compatibility test passed!")


CSR UCS: {'visited_node_indices': [0, 1, 2, 3, 4], 'visited_node_ids': [10, 20, 30, 40, 50], 'discovered_node_indices': [0, 1, 2, 3, 4, 5, 6], 'total_cost': 6, 'execution_time': 9.760000011738157e-05}
CSR A*: {'visited_node_indices': [0, 1, 2, 4, 3], 'visited_node_ids': [10, 20, 30, 50, 40], 'discovered_node_indices': [0, 1, 2, 3, 4, 5, 6], 'total_cost': 6, 'execution_time': 0.00023429999964719173}
CSR compatibility test passed!
